# 02 — Silver → Gold: Dimensions

Builds the three shared dimensions for the Power BI star schema:
`dim_provider` (NPI grain), `dim_hcpcs` (procedure grain),
`dim_geography` (state grain). Each dim provides a cross-filter key
(`npi`, `hcpcs_cd`, `state_abrvtn`) that every hero mart shares, so a
single slicer in Power BI cascades to all five hero pages.

All writes are strict `mode("overwrite")` for deterministic reruns.

In [0]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DecimalType

from utils.secrets import configure_adls_oauth
from utils.paths   import SILVER_PHYSICIAN, GOLD

spark = SparkSession.builder.appName("silver_to_gold_dims").getOrCreate()
spark.conf.set("spark.sql.adaptive.enabled", "true")
configure_adls_oauth(spark, dbutils)

DECIMAL_MONEY = DecimalType(18, 2)

silver = spark.read.format("delta").load(SILVER_PHYSICIAN)

## dim_provider — one row per NPI



In [0]:
dim_provider = (
    silver.groupBy("Rndrng_NPI").agg(
        F.first("Rndrng_Prvdr_Last_Org_Name",    ignorenulls=True).alias("last_org_name"),
        F.first("Rndrng_Prvdr_First_Name",       ignorenulls=True).alias("first_name"),
        F.first("Rndrng_Prvdr_Crdntls",          ignorenulls=True).alias("credentials_raw"),
        F.first("Provider_Tier",                 ignorenulls=True).alias("provider_tier"),
        F.first("Rndrng_Prvdr_Ent_Cd",           ignorenulls=True).alias("ent_cd"),
        F.first("Is_Org",                        ignorenulls=True).alias("is_org"),
        F.first("Rndrng_Prvdr_Type",             ignorenulls=True).alias("specialty"),
        F.first("Rndrng_Prvdr_Mdcr_Prtcptg_Ind", ignorenulls=True).alias("participation"),
        F.first("Is_Participating",              ignorenulls=True).alias("is_participating"),
        F.first("Rndrng_Prvdr_City",             ignorenulls=True).alias("city"),
        F.first("Rndrng_Prvdr_State_Abrvtn",     ignorenulls=True).alias("state_abrvtn"),
        F.first("Rndrng_Prvdr_State_FIPS",       ignorenulls=True).alias("state_fips"),
        F.first("Rndrng_Prvdr_Zip5",             ignorenulls=True).alias("zip5"),
        F.first("RUCA_Int",                      ignorenulls=True).alias("ruca_int"),
        F.first("RUCA_Bucket",                   ignorenulls=True).alias("ruca_bucket"),
        F.first("Is_Rural",                      ignorenulls=True).alias("is_rural"),
        F.sum("Tot_Mdcr_Pymt_Amt").cast(DECIMAL_MONEY).alias("total_mdcr_pymt"),
        F.sum("Tot_Srvcs").alias("total_services"),
        F.countDistinct("HCPCS_Cd").alias("n_distinct_hcpcs"),
    )
    .withColumnRenamed("Rndrng_NPI", "npi")
    .withColumn(
        "display_name",
        F.when(F.col("is_org"), F.col("last_org_name"))
         .otherwise(F.concat_ws(" ", F.col("first_name"), F.col("last_org_name"))),
    )
)

(dim_provider.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD("dim_provider")))
spark.sql(f"OPTIMIZE delta.`{GOLD('dim_provider')}` ZORDER BY (npi)")
print(f"dim_provider written → {GOLD('dim_provider')}")

dim_provider written → abfss://gold@sthealthcareplatdev.dfs.core.windows.net/dim_provider/


## dim_hcpcs — one row per HCPCS code (~10k rows, broadcast-eligible)
Keeps the long `hcpcs_desc` strings OUT of the 9.6M-row fact table.



In [0]:
dim_hcpcs = (
    silver.groupBy("HCPCS_Cd").agg(
        F.first("HCPCS_Desc",     ignorenulls=True).alias("hcpcs_desc"),
        F.first("HCPCS_Drug_Ind", ignorenulls=True).alias("hcpcs_drug_ind"),
        F.first("Is_Drug",        ignorenulls=True).alias("is_drug"),
        F.sum("Tot_Srvcs").alias("national_services"),
        F.sum("Tot_Mdcr_Pymt_Amt").cast(DECIMAL_MONEY).alias("national_mdcr_pymt"),
        F.countDistinct("Rndrng_NPI").alias("n_providers"),
    )
    .withColumn("hcpcs_family", F.substring(F.col("HCPCS_Cd"), 1, 1))
    .withColumnRenamed("HCPCS_Cd", "hcpcs_cd")
)

(dim_hcpcs.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD("dim_hcpcs")))
spark.sql(f"OPTIMIZE delta.`{GOLD('dim_hcpcs')}` ZORDER BY (hcpcs_cd)")
print(f"dim_hcpcs written → {GOLD('dim_hcpcs')}")

dim_hcpcs written → abfss://gold@sthealthcareplatdev.dfs.core.windows.net/dim_hcpcs/


## dim_geography — one row per state (~52 rows, shared cross-filter dim)



In [0]:
dim_geography = (
    silver.groupBy("Rndrng_Prvdr_State_Abrvtn").agg(
        F.first("Rndrng_Prvdr_State_FIPS", ignorenulls=True).alias("state_fips"),
        F.countDistinct("Rndrng_NPI").alias("n_providers"),
        F.sum("Tot_Benes").alias("total_benes_proxy"),
        F.sum("Tot_Mdcr_Pymt_Amt").cast(DECIMAL_MONEY).alias("total_mdcr_pymt"),
        F.avg(F.when(F.col("Is_Rural"), 1).otherwise(0)).alias("rural_share_of_rows"),
        F.avg("Geo_Adj_Factor").alias("avg_geo_adj_factor"),
    )
    .withColumnRenamed("Rndrng_Prvdr_State_Abrvtn", "state_abrvtn")
)

(dim_geography.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD("dim_geography")))
print(f"dim_geography written → {GOLD('dim_geography')}")

dim_geography written → abfss://gold@sthealthcareplatdev.dfs.core.windows.net/dim_geography/
